In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc matplotlib numpy pandas
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Entrenamiento de kernel cuántico

import TutorialFeedback from '@site/src/components/TutorialFeedback';



*Estimación de uso: menos de un minuto en un procesador Eagle r3 (NOTA: Esto es solo una estimación. Su tiempo de ejecución puede variar.)*
## Contexto
Este tutorial muestra cómo construir un `Qiskit pattern` para evaluar entradas en una matriz de kernel cuántico utilizada para clasificación binaria. Para obtener más información sobre los `Qiskit patterns` y cómo se puede utilizar `Qiskit Serverless` para desplegarlos en la nube para una ejecución gestionada, visite nuestra [página de documentación en IBM Quantum&reg; Platform](/guides/serverless).
## Requisitos
Antes de comenzar este tutorial, asegúrate de tener instalado lo siguiente:
- Qiskit SDK v1.0 o posterior, con soporte de [visualización](https://docs.quantum.ibm.com/api/qiskit/visualization)
- Qiskit Runtime v0.22 o posterior (`pip install qiskit-ibm-runtime`)
## Configuración

In [ ]:
# General Imports and helper functions
import urllib.request

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


from qiskit.circuit import Parameter, ParameterVector, QuantumCircuit
from qiskit.circuit.library import unitary_overlap
from qiskit.primitives import StatevectorSampler
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

from qiskit_ibm_runtime import QiskitRuntimeService, Sampler

# Download the dataset (portable across platforms)
urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/qiskit-community/prototype-quantum-kernel-training/main/data/dataset_graph7.csv",
    "dataset_graph7.csv",
)


def visualize_counts(res_counts, num_qubits, num_shots):
    """Visualize the outputs from the Qiskit Sampler primitive."""
    zero_prob = res_counts.get(0, 0.0)
    top_10 = dict(
        sorted(res_counts.items(), key=lambda item: item[1], reverse=True)[
            :10
        ]
    )
    top_10.update({0: zero_prob})
    by_key = dict(sorted(top_10.items(), key=lambda item: item[0]))
    x_vals, y_vals = list(zip(*by_key.items()))
    x_vals = [bin(x_val)[2:].zfill(num_qubits) for x_val in x_vals]
    y_vals_prob = []
    for t in range(len(y_vals)):
        y_vals_prob.append(y_vals[t] / num_shots)
    y_vals = y_vals_prob
    plt.bar(x_vals, y_vals)
    plt.xticks(rotation=75)
    plt.title("Results of sampling")
    plt.xlabel("Measured bitstring")
    plt.ylabel("Probability")
    plt.show()


def get_training_data():
    """Read the training data."""
    df = pd.read_csv("dataset_graph7.csv", sep=",", header=None)
    training_data = df.values[:20, :]
    ind = np.argsort(training_data[:, -1])
    X_train = training_data[ind][:, :-1]

    return X_train

## Small-scale simulator example

In this section, we walk through the four steps of the Qiskit pattern on a seven-qubit instance of the labeling-cosets-with-error problem and evaluate a single kernel matrix entry using the `StatevectorSampler` primitive from Qiskit. A statevector simulator is exact (up to shot noise) and shows us the method end-to-end without consuming QPU time. We then repeat the same instance on real hardware in the hardware example section.

### Step 1: Map classical inputs to a quantum problem

*   Input: Training dataset.
*   Output: Abstract circuit for calculating a kernel matrix entry.

The binary classification problem we aim to solve here is referred to as "[_labeling cosets with error_](https://arxiv.org/abs/2105.03406)." The input training dataset contains a group structure, consisting of two cosets formed by a group and subgroup.
The group is taken to be $G = SU(2)^{\otimes n}$ for qubits, which is the special unitary group of
$2 \times 2$ matrices and has wide applicability in nature; e.g., the Standard Model of particle physics.
We take the (graph-stabilizer) subgroup $S_\text{graph} < G$ with $S_\text{graph} = \langle \{ X_i \otimes _{k:(k,i) \in \mathcal{E}} Z_k\} _{i \in \mathcal{V}} \} \rangle$ for a graph with edges $\mathcal{E}$ and vertices $\mathcal{V}$.
Note that the stabilizers fix a stabilizer state such that $D_s | \psi \rangle = | \psi \rangle,~ \forall s \in S_\text{graph}$.
Finally, we define two left-cosets $C_\pm = c_\pm S_\text{graph}$ by drawing two $c_\pm \in G$ at random.

For more details about the dataset and how it is generated, see [this notebook](https://github.com/qiskit-community/prototype-quantum-kernel-training/blob/main/docs/background/qkernels_and_data_w_group_structure.ipynb) from the [Quantum Kernel Training Toolkit](https://github.com/qiskit-community/prototype-quantum-kernel-training/tree/main).

We create the quantum circuit used to evaluate one entry in the kernel matrix.
The input data is used to determine the rotation angles for the circuit's parametrized gates.
For simplicity, we will use data samples `x1=14` and `x2=19`.

***Note: The dataset used in this tutorial can be downloaded [here](https://github.com/qiskit-community/prototype-quantum-kernel-training/blob/main/data/dataset_graph7.csv).***

In [2]:
# Prepare training data
X_train = get_training_data()

# Empty kernel matrix
num_samples = np.shape(X_train)[0]
kernel_matrix = np.full((num_samples, num_samples), np.nan)

# Prepare feature map for computing overlap
num_features = np.shape(X_train)[1]
num_qubits = int(num_features / 2)
entangler_map = [[0, 2], [3, 4], [2, 5], [1, 4], [2, 3], [4, 6]]
fm = QuantumCircuit(num_qubits)
training_param = Parameter("θ")
feature_params = ParameterVector("x", num_qubits * 2)
fm.ry(training_param, fm.qubits)
for cz in entangler_map:
    fm.cz(cz[0], cz[1])
for i in range(num_qubits):
    fm.rz(-2 * feature_params[2 * i + 1], i)
    fm.rx(-2 * feature_params[2 * i], i)

# Assign tunable parameter to known optimal value and set the data params for first two samples
x1 = 14
x2 = 19
unitary1 = fm.assign_parameters(list(X_train[x1]) + [np.pi / 2])
unitary2 = fm.assign_parameters(list(X_train[x2]) + [np.pi / 2])

# Create the overlap circuit
overlap_circ = unitary_overlap(unitary1, unitary2)
overlap_circ.measure_all()
overlap_circ.draw("mpl", scale=0.6, style="iqp")

<Image src="../docs/images/tutorials/quantum-kernel-training/extracted-outputs/70d6faff-9a56-44bb-b26f-f573a8c90889-0.avif" alt="Output of the previous code cell" />

## Paso 1: Mapear las entradas clásicas a un problema cuántico
*   Entrada: Conjunto de datos de entrenamiento.
*   Salida: Circuito abstracto para calcular una entrada de la matriz de kernel.

Crea el circuito cuántico utilizado para evaluar una entrada en la matriz de kernel. Utilizamos los datos de entrada para determinar los ángulos de rotación de las compuertas parametrizadas del circuito. Usaremos las muestras de datos `x1=14` y `x2=19`.

***Nota: El conjunto de datos utilizado en este tutorial se puede descargar [aquí](https://github.com/qiskit-community/prototype-quantum-kernel-training/blob/main/data/dataset_graph7.csv).***

In [3]:
sampler = StatevectorSampler()

# Execute and get counts
num_shots = 10_000
results = sampler.run([overlap_circ], shots=num_shots).result()
counts = results[0].data.meas.get_int_counts()

# Plot counts
visualize_counts(counts, num_qubits, num_shots)

<Image src="../docs/images/tutorials/quantum-kernel-training/extracted-outputs/step3-small-code-0.avif" alt="Output of the previous code cell" />

![Salida de la celda de código anterior](../docs/images/tutorials/quantum-kernel-training/extracted-outputs/70d6faff-9a56-44bb-b26f-f573a8c90889-0.avif)

## Paso 2: Optimizar el problema para la ejecución en hardware cuántico
*   Entrada: Circuito abstracto, no optimizado para un backend en particular
*   Salida: Circuito y observable objetivo, optimizados para la QPU seleccionada

Utiliza la función `generate_preset_pass_manager` de Qiskit para especificar una rutina de optimización para nuestro circuito con respecto a la QPU en la que planeamos ejecutar el experimento. Establecemos `optimization_level=3`, lo que significa que utilizaremos el gestor de pases predefinido que proporciona el nivel más alto de optimización.

In [4]:
kernel_matrix[x1, x2] = counts.get(0, 0.0) / num_shots
print(f"Fidelity (simulator): {kernel_matrix[x1, x2]}")

Fidelity (simulator): 0.8261


![Salida de la celda de código anterior](../docs/images/tutorials/quantum-kernel-training/extracted-outputs/49607b34-9723-493d-85da-bd97c1351104-0.avif)

## Paso 3: Ejecutar utilizando primitivas de Qiskit
*   Entrada: Circuito objetivo
*   Salida: Distribución de cuasi-probabilidad

Utiliza la primitiva `Sampler` de Qiskit Runtime para reconstruir una distribución de cuasi-probabilidad de los estados obtenidos al muestrear el circuito. Para la tarea de generar una matriz de kernel, estamos particularmente interesados en la probabilidad de medir el estado |0>.

Para esta demostración, ejecutaremos en una QPU con las primitivas de `qiskit-ibm-runtime`. Para ejecutar con las primitivas basadas en `Statevector` de `qiskit`, reemplace el bloque de código que utiliza las primitivas de Qiskit IBM&reg; Runtime con el bloque comentado.

In [5]:
# ------------------------------ Step 1 ------------------------------
# Prepare training data
X_train = get_training_data()

# Empty kernel matrix
num_samples = np.shape(X_train)[0]
kernel_matrix = np.full((num_samples, num_samples), np.nan)

# Prepare feature map for computing overlap
num_features = np.shape(X_train)[1]
num_qubits = int(num_features / 2)
entangler_map = [[0, 2], [3, 4], [2, 5], [1, 4], [2, 3], [4, 6]]
fm = QuantumCircuit(num_qubits)
training_param = Parameter("θ")
feature_params = ParameterVector("x", num_qubits * 2)
fm.ry(training_param, fm.qubits)
for cz in entangler_map:
    fm.cz(cz[0], cz[1])
for i in range(num_qubits):
    fm.rz(-2 * feature_params[2 * i + 1], i)
    fm.rx(-2 * feature_params[2 * i], i)

# Assign tunable parameter to known optimal value and set the data params for first two samples
x1 = 14
x2 = 19
unitary1 = fm.assign_parameters(list(X_train[x1]) + [np.pi / 2])
unitary2 = fm.assign_parameters(list(X_train[x2]) + [np.pi / 2])

# Create the overlap circuit
overlap_circ = unitary_overlap(unitary1, unitary2)
overlap_circ.measure_all()

# ------------------------------ Step 2 ------------------------------
service = QiskitRuntimeService()
# backend = service.least_busy(
#    operational=True, simulator=False, min_num_qubits=overlap_circ.num_qubits
# )
backend = service.backend("ibm_pittsburgh")
print(f"Using backend: {backend.name}")
pm = generate_preset_pass_manager(optimization_level=3, backend=backend)
overlap_ibm = pm.run(overlap_circ)

# ------------------------------ Step 3 ------------------------------
sampler = Sampler(mode=backend)
sampler.options.environment.job_tags = ["TUT_QKT"]

num_shots = 10_000
results = sampler.run([overlap_ibm], shots=num_shots).result()
counts = results[0].data.meas.get_int_counts()
visualize_counts(counts, num_qubits, num_shots)

# ------------------------------ Step 4 ------------------------------
kernel_matrix[x1, x2] = counts.get(0, 0.0) / num_shots
print(f"Fidelity (hardware): {kernel_matrix[x1, x2]}")

Using backend: ibm_pittsburgh


<Image src="../docs/images/tutorials/quantum-kernel-training/extracted-outputs/d2f4f6cf-067e-4d53-aa04-7ca9c803d3e1-1.avif" alt="Output of the previous code cell" />

Fidelity (hardware): 0.7517


![Salida de la celda de código anterior](../docs/images/tutorials/quantum-kernel-training/extracted-outputs/d2f4f6cf-067e-4d53-aa04-7ca9c803d3e1-0.avif)

## Paso 4: Post-procesar y devolver el resultado en el formato clásico deseado

*   Entrada: Distribución de probabilidad
*   Salida: Un solo elemento de la matriz de kernel

Calcula la probabilidad de medir |0> en el circuito de solapamiento y completa la matriz de kernel en la posición correspondiente a las muestras representadas por este circuito de solapamiento en particular (fila 15, columna 20). En esta visualización, el rojo más oscuro indica fidelidades más cercanas a 1.0. Para completar toda la matriz de kernel, necesitamos ejecutar un experimento cuántico para cada entrada.